# Timing diagnostics: where do the ray-step arrival times break?

`timing_diagnostics_job.py` runs LUCiD in `per_photon` mode with physics switched on one process at a time
(geometry only → wall reflection → PMT reflection → Rayleigh → full WCSim 395 nm), for a centred and an
off-centre isotropic source and a barrel diffuser, with `deposit_leg_bound` off and on. Every deposit is saved
with its PMT, time, weight, propagation step and ray.

Two exact tests, using Δ = t − (|PMT − source| − r_PMT) / v (time minus the earliest possible arrival at the
front of the PMT sphere):
- **Causality** (every config): nothing may arrive early, Δ ≥ 0.
- **Direct light** (`geometry`, step 0): 0 ≤ Δ ≤ r_PMT / v (≈ 1.1 ns), the spread being where on the sphere the ray lands.

Deposits with t ≤ 0 are dropped by the `per_photon` hit mode, so a negative-time pathology is invisible here.

In [ ]:
import sys

!{sys.executable} timing_diagnostics_job.py --gpu 2 --out timing_diagnostics.npz

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

r = np.load('timing_diagnostics.npz')
pmt, rs, v = r['pmt_pos'], float(r['sensor_radius']), float(r['speed_of_light'])
configs, sources, legs = [str(c) for c in r['configs']], [str(s) for s in r['sources']], [int(b) for b in r['leg_bound']]
src_pos = dict(zip(sources, r['source_pos']))
TOL = 0.1                                   # ns slack on the exact bounds (float32 times)

def get(cfg, src, lb):
    k = f'{cfg}__{src}__lb{lb}'
    d = {f: r[f'{k}__{f}'] for f in ('w', 't', 'pmt', 'step', 'ray')}
    dist = np.linalg.norm(pmt[d['pmt']] - src_pos[src], axis=1)
    d['dt'] = d['t'] - (dist - rs) / v      # >= 0 for every causal deposit
    return d

print(f'r_PMT = {rs} m, v = {v:.4f} m/ns -> direct window 0 .. {rs / v:.2f} ns')
print(f'configs {configs}\nsources {sources}\nleg bound {legs}')

In [ ]:
# Summary: share of deposited WEIGHT that breaks each test. Direct test only for geometry step 0.
print(f'{"config":10s}{"source":12s}{"lb":>3s}{"weight":>11s}{"acausal":>10s}{"direct out":>12s}'
      f'{"worst dt (ns)":>15s}')
for cfg in configs:
    for src in sources:
        for lb in legs:
            d = get(cfg, src, lb); W = d['w'].sum()
            acausal = d['w'][d['dt'] < -TOL].sum() / W
            s0 = d['step'] == 0
            out = ((d['dt'] < -TOL) | (d['dt'] > rs / v + TOL)) & s0
            direct = f'{d["w"][out].sum() / d["w"][s0].sum():12.2%}' if cfg == 'geometry' else f'{"-":>12s}'
            print(f'{cfg:10s}{src:12s}{lb:3d}{W:11.4g}{acausal:10.2%}{direct}{d["dt"].min():15.2f}')

In [ ]:
# dt distributions, weighted, split into direct (step 0) and later steps; leg bound off vs on.
bins = np.linspace(-20, 200, 441)
for src in sources:
    fig, ax = plt.subplots(1, len(configs), figsize=(4 * len(configs), 3.4), sharey=True)
    for a, cfg in zip(ax, configs):
        for lb, ls in zip(legs, ('-', '--')):
            d = get(cfg, src, lb)
            for m, lab, c in [(d['step'] == 0, 'step 0', 'tab:blue'), (d['step'] > 0, 'step > 0', 'tab:orange')]:
                if m.any():
                    a.hist(d['dt'][m], bins=bins, weights=d['w'][m], histtype='step', ls=ls, color=c,
                           label=f'{lab}, lb {lb}')
        a.axvline(0, color='k', lw=.7); a.set_yscale('log'); a.set_title(cfg)
        a.set_xlabel('t − earliest possible arrival (ns)')
    ax[0].set_ylabel('weighted deposits'); ax[0].legend(fontsize=7)
    fig.suptitle(src, y=1.02); fig.tight_layout(); plt.show()

In [ ]:
# Where the violations land: per PMT, the weight that breaks causality (any config) or the direct
# window (geometry, step 0). Barrel unrolled to (phi, z); caps by (x, y).
CFG, SRC, LB = 'geometry', 'iso_offset', 0           # edit to inspect another set
d = get(CFG, SRC, LB)
bad = (d['dt'] < -TOL) | ((CFG == 'geometry') & (d['step'] == 0) & (d['dt'] > rs / v + TOL))
badw = np.bincount(d['pmt'][bad], weights=d['w'][bad], minlength=len(pmt))
totw = np.bincount(d['pmt'], weights=d['w'], minlength=len(pmt))
frac = np.where(totw > 0, badw / np.where(totw > 0, totw, 1), np.nan)
R = np.hypot(pmt[:, 0], pmt[:, 1]); zc = np.abs(pmt[:, 2]).max() - 0.5
barrel, top, bot = np.abs(pmt[:, 2]) < zc, pmt[:, 2] >= zc, pmt[:, 2] <= -zc
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5), gridspec_kw=dict(width_ratios=[2, 1, 1]))
kw = dict(c=frac, s=3, cmap='viridis', vmin=0, vmax=max(np.nanmax(frac), 1e-6))
ax[0].scatter(np.degrees(np.arctan2(pmt[barrel, 1], pmt[barrel, 0])), pmt[barrel, 2], **{**kw, 'c': frac[barrel]})
ax[0].set(xlabel='phi (deg)', ylabel='z (m)', title='barrel')
for a, m, name in [(ax[1], top, 'top cap'), (ax[2], bot, 'bottom cap')]:
    sc = a.scatter(pmt[m, 0], pmt[m, 1], **{**kw, 'c': frac[m]}); a.set_aspect('equal'); a.set_title(name)
fig.colorbar(sc, ax=ax.tolist(), label='share of PMT weight failing the test')
fig.suptitle(f'{CFG} / {SRC} / leg bound {LB}: {np.nansum(badw) / totw.sum():.2%} of weight fails', y=1.02)
plt.show()